In [0]:
# ================================
# 1. Load Data (from Spark → Pandas)
# ================================

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

df_spark = spark.table("bls_cew.silver.cleaned_data")

df = df_spark.toPandas()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


# ================================
# 2. Basic Info
# ================================

print("\n--- INFO ---")
print(df.info())

print("\n--- DESCRIPTIVE STATS ---")
print(df.describe())


# ================================
# 3. Null Values Check
# ================================

print("\n--- NULL VALUES ---")
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
print(nulls)


# ================================
# 4. Duplicates Check
# ================================

duplicates = df.duplicated().sum()
print("\nDuplicates:", duplicates)


# ================================
# 5. Negative Values Check
# ================================

print("\n--- NEGATIVE VALUES ---")

negative_cols = [
    "annual_avg_emplvl",
    "total_annual_wages",
    "avg_annual_pay"
]

for col in negative_cols:
    count_neg = (df[col] < 0).sum()
    print(f"{col}: {count_neg} negative values")


# ================================
# 6. Sector Analysis (Public vs Private)
# ================================

print("\n--- SECTOR ANALYSIS ---")

sector_stats = df.groupby("sector_type")[[
    "annual_avg_emplvl",
    "avg_annual_pay"
]].mean()

print(sector_stats)


# ================================
# 7. COVID Impact Analysis
# ================================

print("\n--- COVID IMPACT ---")

covid_stats = df.groupby("covid_period")[[
    "annual_avg_emplvl"
]].mean()

print(covid_stats)


# ================================
# 8. KPI Validation
# ================================

print("\n--- KPI VALIDATION ---")

df["calculated_avg_pay"] = df["total_annual_wages"] / df["annual_avg_emplvl"]

comparison = df[[
    "avg_annual_pay",
    "calculated_avg_pay"
]].head(10)

print(comparison)

diff = (df["avg_annual_pay"] - df["calculated_avg_pay"]).abs().mean()
print("Average difference:", diff)


# ================================
# 9. Correlation Analysis
# ================================

print("\n--- CORRELATION ---")

corr = df[[
    "annual_avg_emplvl",
    "total_annual_wages",
    "avg_annual_pay"
]].corr()

print(corr)


# ================================
# 11. Trend Over Time
# ================================

print("\n--- TREND OVER TIME ---")

trend = df.groupby("year")[[
    "annual_avg_emplvl"
]].mean()

print(trend)



# ================================
# 12. Final Validation Summary
# ================================

print("\n--- FINAL VALIDATION ---")

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"Duplicates: {duplicates}")
print(f"Avg Pay Difference: {diff}")

print("\nValidation completed successfully ✅")